# **Introduction to MCP**

## 1. Overview

### What is the Model Context Protocol (MCP)?

> Protocolo criado pela Anthropic para facilitar a integração entre ferramentas e LLMs 


The **Model Context Protocol (MCP)** is a standardized way for applications to provide **external context and capabilities**—such as data sources, APIs, and executable tools—to large language models (LLMs).

Instead of building custom integrations for every tool or dataset, MCP defines a **common interface** that allows models to discover and use these capabilities dynamically. This makes AI systems more modular, extensible, and easier to maintain.

---

### Why MCP Exists

Modern LLM applications often need to interact with:

* Databases
* APIs
* Local files
* External services

Without a standard like MCP, each integration must be implemented manually, leading to **fragile and hard-to-scale systems**.

MCP addresses this by introducing a **protocol layer** that:

* Decouples the model from specific tools
* Enables reusable integrations
* Standardizes how context is provided to the model

---

### Core Architecture

At a high level, MCP involves three main components:

* **Host**
  The application that uses the LLM (e.g., a chatbot, IDE, or agent system).

* **MCP Client**
  A component within the host that communicates using the MCP protocol. It manages connections to servers and handles tool execution requests.

* **MCP Server**
  A service that exposes capabilities to the model, such as tools, data resources, and prompt templates.

These components work together to allow the model to access external functionality in a structured way.

---

### Capabilities Exposed by MCP

MCP servers provide three primary types of capabilities:

* **Tools**
  Functions the model can call (e.g., querying a database, calling an API).

* **Resources**
  Data the model can read (e.g., files, documents, structured records).

* **Prompts**
  Predefined templates that guide model behavior in specific tasks.

---

### How MCP Works (Conceptually)

1. The host application connects to one or more MCP servers via the MCP client
2. Servers advertise available tools, resources, and prompts
3. The model receives this structured context
4. During execution, the model may decide to use a tool
5. The MCP client sends the request to the server and returns the result
6. The model continues reasoning with the new information

---

### MCP vs Traditional Tool Integration

Traditional approaches require hardcoding each tool integration into the application. In contrast, MCP enables:

* **Dynamic discovery of tools**
* **Standardized interfaces across systems**
* **Separation between tool providers and model logic**

This makes MCP particularly valuable for building scalable AI systems.

---

### Key Takeaway

MCP is best understood as a **protocol for connecting LLMs to the outside world**.

It allows models to move beyond static text generation and interact with real systems in a **structured, reusable, and extensible way**.


## 2. MCP in OpenAI Agents SDK

1. Create a Client
2. Have it spawn a server
3. Collect the tools that the server can use


In [1]:
# Setup 

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

load_dotenv()

True

**Trying with the ```Fetch mcp-server```**

Everything begins with parameters, the way of describing the MCP server.

The stuff in the dictionary of parameters is something which will be run at the command line that will spawn this MCP server.

In [2]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

We need to tell the OpenAI Agent SDK that we want it to create an MCP client, spawn the MCP server, run it, and ask what tools it can provide us.

In [3]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.session.list_tools()
# Stdio = Standard input output: the most common connection for MCP servers running locally
# For remote servers, we use SSE (HTTPS protocol)
fetch_tools.tools

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get 

Look at the tool description. It's amazing.

This one was a Python based MCP server, executed calling uvx and passing the name registered on PyPI. Now we're going to run a JavaScript based MCP server using node and the npx command.

**We're going to use the ```Playwright MCP server.```**

In [4]:
playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.session.list_tools()

playwright_tools.tools

[Tool(name='browser_close', title=None, description='Close the page', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {}, 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Close browser', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execution=None),
 Tool(name='browser_resize', title=None, description='Resize the browser window', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {'width': {'type': 'number', 'description': 'Width of the browser window'}, 'height': {'type': 'number', 'description': 'Height of the browser window'}}, 'required': ['width', 'height'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Resize browser window', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execut

Anthropic's ```server-filesystem``` server. This is an official MCP server that exposes filesystem operations as tools to an AI agent.

In [5]:
os.makedirs("sandbox", exist_ok=True)

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
# This line is constructing a safe, absolute filesystem path to a directory named "sandbox" inside the current working directory.
# os.getcwd(): Returns the current working directory (CWD) — the directory where your Python process is running.
# os.path.join(os.getcwd(), "sandbox"): Appends "sandbox" to the current directory in an OS-safe way.
# os.path.abspath(...): removes . (current directory) or .. (parent directory)

files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
# path is passed to the MCP filesystem server, meaning: “The server is only allowed to operate inside this directory.”
# it defines the maximum permission scope for file operations

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.session.list_tools()

file_tools.tools


[Tool(name='read_file', title='Read File (Deprecated)', description='Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}, 'required': ['path']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'content': {'type': 'string'}}, 'required': ['content'], 'additionalProperties': False}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None, execution=ToolExecution(taskSupport='forbidden')),
 Tool(name='read_text_file', title='Read Text File', description="Read the complete contents of a fil

**In MCP terminology:**

- **Host** → your Python app (this script)
- **Client** → MCPServerStdio (handles communication)
- **Server** → @modelcontextprotocol/server-filesystem
- **Tools** → filesystem operations exposed by the server

**Important architectural detail**

The server is sandboxed to:

```sandbox_path```

This means:

- It cannot access your entire filesystem
- It is restricted to that directory (security boundary)

## 3. Bringing on the Agent with Tools

**I had to:**

```npx @playwright/mcp install-browser chrome-for-testing```

```npx playwright install-deps```

```npx playwright open https://cnn.com```

In [6]:
playwright_params = {
    "command": "npx",
    "args": [
        "@playwright/mcp@latest",
        "--browser=chromium" # Forcing it to open with chromium
    ]
}

In [7]:
instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task, 
including accepting all cookies and clicking 'not now' as
appropriate to get to the content you need. If one website isn't fruitful, try another. 
Be persistent until you have solved your assignment,
trying different options and sites as needed.
When you need to write files, you do that inside the sandbox folder only.
"""


async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model="gpt-4.1-mini",
            mcp_servers=[mcp_server_files, mcp_server_browser]
            )
        with trace("investigate"):
            result = await Runner.run(agent, "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md")
            print(result.final_output)



I have found a great recipe for Banoffee Pie and summarized it in markdown format in the file "banoffee.md". If you'd like, I can show you the contents or help with anything else!


#### Trace

https://platform.openai.com/logs?api=traces

#### MCP Marketplaces

https://mcp.so

https://glama.ai/mcp

https://smithery.ai/

https://huggingface.co/blog/LLMhacker/top-11-essential-mcp-libraries

HuggingFace great community article: https://huggingface.co/blog/Kseniase/mcp